In [1]:
# ==============================================================================
# 01. Data Audit & Feature Engineering
# ==============================================================================
import pandas as pd
import numpy as np
import re
import os

In [2]:
# Create necessary output directories if they don't exist
os.makedirs('../data/processed', exist_ok=True)

In [3]:
# 1. Load Raw Data
print("Loading datasets...")
app_df = pd.read_csv('../data/raw/pd_applications.csv')
bureau_df = pd.read_csv('../data/raw/pd_bureau_history.csv')

Loading datasets...


In [5]:
# 2. Data Audit & Cleaning
app_df = app_df.drop_duplicates(subset=['app_id'], keep='last').copy()
app_df['home_ownership'] = app_df['home_ownership'].astype(str).str.strip().str.upper()

def clean_emp_length(val):
    if pd.isna(val): return np.nan
    val_str = str(val).strip()
    if '< 1' in val_str: return 0.5
    if '> 10+' in val_str: return 10.0
    nums = re.findall(r'\d+', val_str)
    return  float(nums[0]) if nums else np.nan

app_df['emp_length_years'] = app_df['employment_length_raw'].apply(clean_emp_length)

def clean_util(val):
    if pd.isna(val): return np.nan
    val_str = str(val).replace('%', '').strip()
    try: return float(val_str)
    except ValueError: return np.nan

app_df['revol_util_clean'] = app_df['revolving_utilization_raw'].apply(clean_util)
app_df['annual_income'] = app_df['annual_income'].apply(
    lambda x: np.nan if (pd.isna(x) or x <= 0 or x >= 9999999) else x
)

In [8]:
# 3. Feature Engineering: DTI and Bureau Aggregations
app_df['monthly_income'] = app_df['annual_income'] / 12.0
app_df['dti_engineered'] = app_df['monthly_debt_payments'] / (app_df['monthly_income'] + 1e-6)

bureau_agg = bureau_df.groupby('app_id').agg(
    bureau_num_accounts=('bureau_id', 'count'),
    bureau_active_accounts=('account_status', lambda x: (x == 'Active').sum()),
    bureau_total_credit_limit=('credit_limit', 'sum'),
    bureau_total_current_balance=('current_balance', 'sum'),
    bureau_max_months_past_due=('months_past_due', 'max'),
    bureau_avg_account_age=('account_age_months', 'mean')
).reset_index()

df_model = pd.merge(app_df, bureau_agg, on='app_id', how='left')
bureau_cols = [col for col in bureau_agg.columns if col != 'app_id']
df_model[bureau_cols] = df_model[bureau_cols].fillna(0)

# 4. Censoring & Target Definition
# Exclude Current/Late loans to prevent unobserved outcomes from leaking into training
resolved_statuses = ['Fully Paid', 'Charged Off']
df_resolved = df_model[df_model['loan_status'].isin(resolved_statuses)].copy()

df_resolved['target'] = (df_resolved['loan_status'] == 'Charged Off').astype(int)
print(f"Modeling Population (Resolved Loans): {len(df_resolved)}")
print(f"Portfolio Default Rate: {df_resolved['target'].mean():.2%}")

# 5. Export Model-Ready Data
df_resolved.to_csv('../data/processed/model_ready_joined_dataset.csv', index=False)
print("Data exported to '../data/processed/model_ready_joined_dataset.csv'")

Modeling Population (Resolved Loans): 14058
Portfolio Default Rate: 11.40%
Data exported to '../data/processed/model_ready_joined_dataset.csv'
